# exp289 fault-aware transductive geological potential — inference disabled

Stage 0 creates a diagnostic risk readout, not a TVT prediction. Inference and
submission remain fail-closed until Stage 0 passes and a separately approved
Stage 1 direct MAP implementation satisfies its preregistered guards.

## Contents
1. Imports and configuration
2. Disabled inference contract
3. Fail-closed execution

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import os
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import yaml

EXPERIMENT_NAME = "exp289_fault_aware_transductive_geological_potential"


def in_notebook_runtime() -> bool:
    try:
        return get_ipython() is not None  # type: ignore[name-defined]
    except NameError:
        return False


EXECUTE_NOTEBOOK = os.environ.get("EXP289_IMPORT_ONLY", "0") != "1" and in_notebook_runtime()


def get_nested(config: Mapping[str, Any], dotted_key: str, default: Any = None) -> Any:
    current: Any = config
    for part in dotted_key.split("."):
        if not isinstance(current, Mapping) or part not in current:
            return default
        current = current[part]
    return current


def load_config() -> dict[str, Any]:
    start = Path.cwd()
    candidates = [start / "config.yaml"]
    for parent in (start, *start.parents):
        candidates.append(parent / "experiments" / EXPERIMENT_NAME / "config.yaml")
    for path in candidates:
        if not path.exists():
            continue
        value = yaml.safe_load(path.read_text()) or {}
        if get_nested(value, "experiment.name") == EXPERIMENT_NAME:
            return value
    raise FileNotFoundError("exp289 config.yaml not found")

## 2. Disabled inference contract

In [ ]:
def validate_disabled_inference(config: Mapping[str, Any]) -> dict[str, Any]:
    contract = {
        "route": get_nested(config, "experiment.route"),
        "stage0_status": get_nested(config, "stages.stage0.implementation_status"),
        "stage1_status": get_nested(config, "stages.stage1.implementation_status"),
        "inference_enabled": bool(get_nested(config, "inference.enabled")),
        "create_submission": bool(get_nested(config, "inference.create_submission")),
    }
    if contract["route"] != "pf_beam":
        raise ValueError("exp289 route must remain pf_beam")
    if contract["inference_enabled"] or contract["create_submission"]:
        raise ValueError("exp289 inference must remain disabled before Stage 1 approval")
    return contract


def fail_closed() -> None:
    raise RuntimeError(
        "exp289 currently implements only the Stage 0 fault-topology association "
        "readout. No TVT prediction or submission may be created before the Stage 0 "
        "guard passes and Stage 1 receives separate user approval."
    )

## 3. Fail-closed execution

In [ ]:
config = load_config()
contract = validate_disabled_inference(config)
print("Experiment:", EXPERIMENT_NAME)
print("Inference contract:", contract)
if EXECUTE_NOTEBOOK:
    fail_closed()